In [ ]:
# %% [Cell 1] Imports & config
import hashlib
import re
import uuid
from datetime import datetime

import ipywidgets as widgets
from IPython.display import display, clear_output
from google.cloud import bigquery
from pypdf import PdfReader

PROJECT_ID = "prj-dbi-prd-1"
DATASET_ID = "ds_dbi_digitalmedia_automation"
MANIFEST_TABLE = f"{PROJECT_ID}.{DATASET_ID}.sdi_competitiveOffers_bronze_uploadManifest_weekly"
BRONZE_TABLE = f"{PROJECT_ID}.{DATASET_ID}.sdi_competitiveOffers_bronze_pdfPages_weekly"

bq = bigquery.Client(project=PROJECT_ID)

# Manifest table DDL (run once, not part of the notebook flow):
# CREATE TABLE IF NOT EXISTS `sdi_competitiveOffers_bronze_uploadManifest_weekly` (
#   report_date DATE,
#   run_id STRING,
#   file_name STRING,
#   file_hash STRING,
#   status STRING,         -- 'active' | 'superseded' | 'cancelled'
#   uploaded_at TIMESTAMP,
#   uploaded_by STRING
# )


# %% [Cell 2] Helpers: parse date, hash file, check manifest, write manifest row

def parse_date_from_filename(filename: str):
    """MCI's convention: '..._MMDDYY.pdf' e.g. Competitive_Offer_Report_050826.pdf"""
    m = re.search(r"_(\d{2})(\d{2})(\d{2})\.pdf$", filename, re.IGNORECASE)
    if not m:
        return None
    mm, dd, yy = m.groups()
    return datetime.strptime(f"20{yy}-{mm}-{dd}", "%Y-%m-%d").date()


def verify_date_from_content(pdf_bytes: bytes, expected_date):
    """Cross-check the filename-derived date against text actually inside the PDF
    (CI Headlines page reliably contains 'COMPETITIVE INTELLIGENCE HEADLINES <Month D, YYYY>').
    Cheap: only reads a couple of pages, not the whole deck."""
    from io import BytesIO
    reader = PdfReader(BytesIO(pdf_bytes))
    for page in reader.pages[:6]:
        text = page.extract_text() or ""
        m = re.search(r"([A-Z][a-z]+ \d{1,2}, 20\d{2})", text)
        if m:
            found = datetime.strptime(m.group(1), "%B %d, %Y").date()
            return found == expected_date, found
    return None, None  # couldn't verify — not necessarily a problem, just flag as unverified


def file_hash(pdf_bytes: bytes) -> str:
    return hashlib.sha256(pdf_bytes).hexdigest()


def get_active_manifest_row(report_date):
    query = f"""
        SELECT run_id, file_name, file_hash, uploaded_at
        FROM `{MANIFEST_TABLE}`
        WHERE report_date = @report_date AND status = 'active'
        ORDER BY uploaded_at DESC
        LIMIT 1
    """
    job = bq.query(
        query,
        job_config=bigquery.QueryJobConfig(
            query_parameters=[bigquery.ScalarQueryParameter("report_date", "DATE", report_date)]
        ),
    )
    rows = list(job.result())
    return rows[0] if rows else None


def write_manifest_row(report_date, run_id, file_name, hash_, status):
    row = {
        "report_date": report_date.isoformat(),
        "run_id": run_id,
        "file_name": file_name,
        "file_hash": hash_,
        "status": status,
        "uploaded_at": datetime.utcnow().isoformat(),
        "uploaded_by": "notebook1",
    }
    bq.insert_rows_json(MANIFEST_TABLE, [row])


def supersede_active_run(report_date):
    """Flip the currently-active run for this report_date to 'superseded'.
    Never deletes — bronze stays immutable, downstream just stops looking at it."""
    query = f"""
        UPDATE `{MANIFEST_TABLE}`
        SET status = 'superseded'
        WHERE report_date = @report_date AND status = 'active'
    """
    bq.query(
        query,
        job_config=bigquery.QueryJobConfig(
            query_parameters=[bigquery.ScalarQueryParameter("report_date", "DATE", report_date)]
        ),
    ).result()


# %% [Cell 3] Placeholder — hand off to the actual extraction pipeline (built separately)

def run_extraction_pipeline(run_id: str, report_date, pdf_bytes: bytes, file_name: str):
    print(f"[{run_id}] Starting extraction for {file_name} ({report_date}) ...")
    # -> page-type classification, dual text+Gemini extraction,
    #    writes to BRONZE_TABLE and the per-shape silver tables, tagged with run_id
    print(f"[{run_id}] Done.")


# %% [Cell 4] Upload widget + dedup check + decision UI

upload_widget = widgets.FileUpload(accept=".pdf", multiple=False, description="Upload PDF")
output = widgets.Output()
display(upload_widget, output)


def proceed(report_date, run_id, file_name, hash_, pdf_bytes, mode):
    with output:
        if mode == "cancel":
            print("Cancelled — nothing written.")
            return
        if mode == "replace":
            supersede_active_run(report_date)
        write_manifest_row(report_date, run_id, file_name, hash_, status="active")
        run_extraction_pipeline(run_id, report_date, pdf_bytes, file_name)


def handle_upload(change):
    output.clear_output()
    uploaded = list(upload_widget.value.values())[0] if hasattr(upload_widget.value, "values") else upload_widget.value[0]
    file_name = uploaded["metadata"]["name"] if "metadata" in uploaded else uploaded.name
    pdf_bytes = uploaded["content"] if "content" in uploaded else uploaded.content

    with output:
        report_date = parse_date_from_filename(file_name)
        if report_date is None:
            print(f"Couldn't parse a report date from filename '{file_name}' — check naming convention.")
            return

        match, found_date = verify_date_from_content(pdf_bytes, report_date)
        if match is False:
            print(f"Warning: filename implies {report_date}, but content suggests {found_date}. Proceeding with content date.")
            report_date = found_date

        hash_ = file_hash(pdf_bytes)
        existing = get_active_manifest_row(report_date)
        run_id = str(uuid.uuid4())

        if existing is None:
            print(f"No existing data for {report_date}. Proceeding.")
            proceed(report_date, run_id, file_name, hash_, pdf_bytes, mode="new")
            return

        if existing["file_hash"] == hash_:
            print(f"This exact file was already processed on {existing['uploaded_at']} (run {existing['run_id']}). Skipping.")
            return

        print(f"Data for {report_date} already exists (run {existing['run_id']}, file '{existing['file_name']}').")
        print("Choose how to proceed:")

        replace_btn = widgets.Button(description="Replace", button_style="danger")
        append_btn = widgets.Button(description="Append", button_style="warning")
        cancel_btn = widgets.Button(description="Cancel", button_style="")
        display(widgets.HBox([replace_btn, append_btn, cancel_btn]))

        replace_btn.on_click(lambda b: proceed(report_date, run_id, file_name, hash_, pdf_bytes, "replace"))
        # NOTE on append: this leaves two 'active' runs for the same report_date.
        # Decide before wiring this up for real: should gold then show both, average them,
        # or should append actually mean something else in your workflow (e.g. a corrected
        # re-issue from MCI, in which case it should probably behave like replace)?
        append_btn.on_click(lambda b: proceed(report_date, run_id, file_name, hash_, pdf_bytes, "append"))
        cancel_btn.on_click(lambda b: proceed(report_date, run_id, file_name, hash_, pdf_bytes, "cancel"))


upload_widget.observe(handle_upload, names="value")